<a href="https://colab.research.google.com/github/RizkyAditya007/Pemrograman-berorientasi-objek/blob/main/jobsheet_11(PBO)_Rizky_Aditya.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Langkah 1: Persiapan Awal (Konfigurasi & Setup Database)

install library

In [7]:

!pip install streamlit pyngrok -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 103.9 MB/s eta 0:00:00


konfigurasi & database

In [16]:
%%writefile konfigurasi.py
import os

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
NAMA_DB = 'pengeluaran_harian.db'
DB_PATH = os.path.join(BASE_DIR, NAMA_DB)
KATEGORI_PENGELUARAN = [
    "Makanan", "Transportasi", "Hiburan", "Tagihan",
    "Belanja", "Kesehatan", "Pendidikan", "Lainnya"
]
KATEGORI_DEFAULT = "Lainnya"

Writing konfigurasi.py


In [17]:
%%writefile setup_db_pengeluaran.py

import sqlite3, os
from konfigurasi import DB_PATH

def setup_database():
    conn = None
    try:
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        sql = """
        CREATE TABLE IF NOT EXISTS transaksi (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            deskripsi TEXT NOT NULL,
            jumlah REAL NOT NULL CHECK(jumlah > 0),
            kategori TEXT,
            tanggal DATE NOT NULL
        );"""
        cursor.execute(sql)
        conn.commit()
        return True
    except sqlite3.Error as e:
        print(f"Error: {e}"); return False
    finally:
        if conn: conn.close()

if __name__ == "__main__":
    setup_database()

Writing setup_db_pengeluaran.py


#Langkah	2:	Modul	Akses	Database (database.py)

In [18]:
%%writefile database.py
import sqlite3
import pandas as pd
from konfigurasi import DB_PATH

def get_db_connection():
    """Membuka dan mengembalikan koneksi baru ke database SQLite."""
    try:
        conn = sqlite3.connect(DB_PATH, timeout=10, detect_types=sqlite3.PARSE_DECLTYPES)
        conn.row_factory = sqlite3.Row
        return conn
    except sqlite3.Error as e:
        print(f"ERROR [database.py] Koneksi DB gagal: {e}")
        return None

def execute_query(query: str, params: tuple = None):
    """Menjalankan query non-SELECT (INSERT/UPDATE/DELETE). Mengembalikan lastrowid."""
    conn = get_db_connection()
    if not conn:
        return None
    last_id = None
    try:
        cursor = conn.cursor()
        if params:
            cursor.execute(query, params)
        else:
            cursor.execute(query)
        conn.commit()
        last_id = cursor.lastrowid
        return last_id
    except sqlite3.Error as e:
        print(f"ERROR [database.py] Query gagal: {e} | Query: {query[:60]}")
        conn.rollback()
        return None
    finally:
        if conn:
            conn.close()

def fetch_query(query: str, params: tuple = None, fetch_all: bool = True):
    """Menjalankan query SELECT dan mengembalikan hasil."""
    conn = get_db_connection()
    if not conn:
        return None
    try:
        cursor = conn.cursor()
        if params:
            cursor.execute(query, params)
        else:
            cursor.execute(query)
        result = cursor.fetchall() if fetch_all else cursor.fetchone()
        return result
    except sqlite3.Error as e:
        print(f"ERROR [database.py] Fetch gagal: {e} | Query: {query[:60]}")
        return None
    finally:
        if conn:
            conn.close()

def get_dataframe(query: str, params: tuple = None):
    """Menjalankan query SELECT dan mengembalikan DataFrame Pandas."""
    conn = get_db_connection()
    if not conn:
        return pd.DataFrame()
    try:
        df = pd.read_sql_query(query, conn, params=params)
        return df
    except Exception as e:
        print(f"ERROR [database.py] Gagal baca ke DataFrame: {e}")
        return pd.DataFrame()
    finally:
        if conn:
            conn.close()

def setup_database_initial():
    """Memastikan tabel transaksi ada di database."""
    print(f"Memeriksa/membuat tabel di database: {DB_PATH}")
    conn = get_db_connection()
    if not conn:
        return False
    try:
        cursor = conn.cursor()
        sql_create_table = """
        CREATE TABLE IF NOT EXISTS transaksi (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            deskripsi TEXT NOT NULL,
            jumlah REAL NOT NULL CHECK(jumlah > 0),
            kategori TEXT,
            tanggal DATE NOT NULL
        );"""
        cursor.execute(sql_create_table)
        conn.commit()
        print("-> Tabel 'transaksi' siap.")
        return True
    except sqlite3.Error as e:
        print(f"Error SQLite saat setup tabel: {e}")
        return False
    finally:
        if conn:
            conn.close()

Overwriting database.py


#Langkah 3: Modul Model Data (model.py)

In [19]:
%%writefile model.py
import datetime

class Transaksi:
    """Merepresentasikan satu entitas transaksi pengeluaran."""

    def __init__(self, deskripsi: str, jumlah: float, kategori: str,
                 tanggal, id_transaksi: int = None):
        self.id = id_transaksi
        self.deskripsi = str(deskripsi) if deskripsi else "Tanpa Deskripsi"

        try:
            jumlah_float = float(jumlah)
            self.jumlah = jumlah_float if jumlah_float > 0 else 0.0
            if jumlah_float <= 0:
                print(f"Peringatan: Jumlah '{jumlah}' harus positif.")
        except (ValueError, TypeError):
            self.jumlah = 0.0
            print(f"Peringatan: Jumlah '{jumlah}' tidak valid.")

        self.kategori = str(kategori) if kategori else "Lainnya"

        if isinstance(tanggal, datetime.date):
            self.tanggal = tanggal
        elif isinstance(tanggal, str):
            try:
                self.tanggal = datetime.datetime.strptime(tanggal, "%Y-%m-%d").date()
            except ValueError:
                self.tanggal = datetime.date.today()
                print(f"Peringatan: Format tanggal '{tanggal}' salah, gunakan YYYY-MM-DD.")
        else:
            self.tanggal = datetime.date.today()
            print(f"Peringatan: Tipe tanggal '{type(tanggal)}' tidak valid.")

    def __repr__(self) -> str:
        return (f"Transaksi(ID:{self.id}, Tgl:{self.tanggal.strftime('%Y-%m-%d')}, "
                f"Jml:{self.jumlah:.0f}, Kat:'{self.kategori}', Desc:'{self.deskripsi}')")

    def to_dict(self) -> dict:
        return {
            "deskripsi": self.deskripsi,
            "jumlah": self.jumlah,
            "kategori": self.kategori,
            "tanggal": self.tanggal.strftime("%Y-%m-%d")
        }

Writing model.py


#Langkah	4:	Modul	Manajer	Anggaran (manajer_anggaran.py)

In [24]:
%%writefile manajer_anggaran.py
import datetime
import pandas as pd
from model import Transaksi
import database

class AnggaranHarian:
    _db_setup_done = False

    def __init__(self):
        if not AnggaranHarian._db_setup_done:
            print("[AnggaranHarian] Melakukan pengecekan/setup database awal...")
            if database.setup_database_initial():
                AnggaranHarian._db_setup_done = True
                print("[AnggaranHarian] Database siap.")
            else:
                print("[AnggaranHarian] KRITIKAL: Setup database awal GAGAL!")

    def tambah_transaksi(self, transaksi: Transaksi) -> bool:
        if not isinstance(transaksi, Transaksi) or transaksi.jumlah <= 0:
            return False
        sql = "INSERT INTO transaksi (deskripsi, jumlah, kategori, tanggal) VALUES (?, ?, ?, ?)"
        params = (
            transaksi.deskripsi,
            transaksi.jumlah,
            transaksi.kategori,
            transaksi.tanggal.strftime("%Y-%m-%d")
        )
        last_id = database.execute_query(sql, params)
        if last_id is not None:
            transaksi.id = last_id
            return True
        return False

    def get_dataframe_transaksi(self, filter_tanggal: datetime.date = None) -> pd.DataFrame:
        # ← Tambahkan id di SELECT agar bisa ditampilkan dan dipakai hapus
        query = "SELECT id, tanggal, kategori, deskripsi, jumlah FROM transaksi"
        params = None
        if filter_tanggal:
            query += " WHERE tanggal = ?"
            params = (filter_tanggal.strftime("%Y-%m-%d"),)
        query += " ORDER BY tanggal DESC, id DESC"
        df = database.get_dataframe(query, params=params)
        if not df.empty:
            df['Jumlah (Rp)'] = df['jumlah'].map(lambda x: f"Rp {x or 0:,.0f}".replace(",", "."))
            # Tampilkan kolom ID agar user tahu nomor yang harus dimasukkan
            df = df[['id', 'tanggal', 'kategori', 'deskripsi', 'Jumlah (Rp)']]
            df = df.rename(columns={'id': 'ID', 'tanggal': 'Tanggal',
                                    'kategori': 'Kategori', 'deskripsi': 'Deskripsi'})
        return df

    def hitung_total_pengeluaran(self, tanggal: datetime.date = None) -> float:
        sql = "SELECT SUM(jumlah) FROM transaksi"
        params = None
        if tanggal:
            sql += " WHERE tanggal = ?"
            params = (tanggal.strftime("%Y-%m-%d"),)
        result = database.fetch_query(sql, params=params, fetch_all=False)
        if result and result[0] is not None:
            return float(result[0])
        return 0.0

    def get_pengeluaran_per_kategori(self, tanggal: datetime.date = None) -> dict:
        hasil = {}
        sql = "SELECT kategori, SUM(jumlah) FROM transaksi"
        params = []
        if tanggal:
            sql += " WHERE tanggal = ?"
            params.append(tanggal.strftime("%Y-%m-%d"))
        sql += " GROUP BY kategori HAVING SUM(jumlah) > 0 ORDER BY SUM(jumlah) DESC"
        rows = database.fetch_query(sql, params=tuple(params) if params else None, fetch_all=True)
        if rows:
            for row in rows:
                kategori = row['kategori'] if row['kategori'] else "Lainnya"
                jumlah = float(row[1]) if row[1] is not None else 0.0
                hasil[kategori] = jumlah
        return hasil

    def hapus_transaksi(self, id_transaksi: int) -> bool:
        """Menghapus transaksi berdasarkan ID."""
        # Cek dulu apakah ID ada
        cek = database.fetch_query(
            "SELECT id FROM transaksi WHERE id = ?",
            params=(id_transaksi,),
            fetch_all=False
        )
        if not cek:
            print(f"[hapus_transaksi] ID {id_transaksi} tidak ditemukan di database.")
            return False
        sql = "DELETE FROM transaksi WHERE id = ?"
        result = database.execute_query(sql, (id_transaksi,))
        return result is not None

Overwriting manajer_anggaran.py


#Langkah	5:	Aplikasi	Utama	Streamlit (main_app.py)

In [25]:
%%writefile main_app.py
import streamlit as st
import datetime
import pandas as pd

def format_rp(angka):
    try:
        return f"Rp {angka or 0:,.0f}".replace(",", ".")
    except:
        return "Rp 0"

try:
    from model import Transaksi
    from manajer_anggaran import AnggaranHarian
    from konfigurasi import KATEGORI_PENGELUARAN
except ImportError as e:
    st.error(f"Gagal mengimpor modul: {e}.")
    st.stop()

st.set_page_config(page_title="Catatan Pengeluaran", layout="wide",
                   initial_sidebar_state="expanded")

@st.cache_resource
def get_anggaran_manager():
    return AnggaranHarian()

# ── Halaman: Tambah ────────────────────────────────────────────────────────────
def halaman_input(anggaran):
    st.header("Tambah Pengeluaran Baru")
    with st.form("form_transaksi_baru", clear_on_submit=True):
        col1, col2 = st.columns([3, 1])
        with col1:
            deskripsi = st.text_input("Deskripsi*", placeholder="Contoh: Makan siang")
        with col2:
            kategori = st.selectbox("Kategori*:", KATEGORI_PENGELUARAN)
        col3, col4 = st.columns([1, 1])
        with col3:
            jumlah = st.number_input("Jumlah (Rp)*:", min_value=0.01, step=1000.0,
                                     format="%.0f", value=None, placeholder="Contoh: 25000")
        with col4:
            tanggal = st.date_input("Tanggal*:", value=datetime.date.today())
        submitted = st.form_submit_button("Simpan Transaksi")
        if submitted:
            if not deskripsi:
                st.warning("Deskripsi wajib diisi!")
            elif jumlah is None or jumlah <= 0:
                st.warning("Jumlah wajib diisi dan harus lebih dari 0!")
            else:
                tx = Transaksi(deskripsi, float(jumlah), kategori, tanggal)
                if anggaran.tambah_transaksi(tx):
                    st.success("Transaksi berhasil disimpan!")
                    st.cache_data.clear()
                    st.rerun()
                else:
                    st.error("Gagal menyimpan transaksi.")

# ── Halaman: Riwayat ───────────────────────────────────────────────────────────
def halaman_riwayat(anggaran):
    st.subheader("Riwayat Semua Transaksi")

    if st.button("Refresh Riwayat"):
        st.cache_data.clear()
        st.rerun()

    df = anggaran.get_dataframe_transaksi()

    if df is None or df.empty:
        st.info("Belum ada transaksi.")
    else:
        # Tampilkan tabel dengan kolom ID yang terlihat jelas
        st.dataframe(df, use_container_width=True, hide_index=True,
                     column_config={"ID": st.column_config.NumberColumn("ID", width="small")})

    # ── Hapus Transaksi ────────────────────────────────────────────────────────
    st.divider()
    st.subheader("Hapus Transaksi")

    if df is not None and not df.empty:
        # Ambil daftar ID yang valid dari tabel untuk dijadikan pilihan
        daftar_id = df['ID'].tolist()
        st.caption(f"ID yang tersedia: {daftar_id}")

    id_hapus = st.number_input(
        "Masukkan ID Transaksi (lihat kolom ID di tabel atas):",
        min_value=1, step=1, value=None,
        placeholder="Contoh: 1"
    )

    if st.button("Hapus Transaksi Terpilih", type="primary"):
        if id_hapus is None:
            st.warning("Masukkan ID transaksi terlebih dahulu.")
        else:
            st.session_state["konfirmasi_hapus"] = True
            st.session_state["id_akan_dihapus"] = int(id_hapus)

    # Blok konfirmasi dua langkah
    if st.session_state.get("konfirmasi_hapus", False):
        id_terpilih = st.session_state.get("id_akan_dihapus")
        st.warning(f"Yakin ingin menghapus transaksi ID **{id_terpilih}**? Tindakan ini tidak bisa dibatalkan.")
        col_ya, col_tidak, _ = st.columns([1, 1, 4])
        with col_ya:
            if st.button("✓ Konfirmasi Hapus", type="primary"):
                berhasil = anggaran.hapus_transaksi(id_terpilih)
                if berhasil:
                    st.success(f"Transaksi ID {id_terpilih} berhasil dihapus!")
                else:
                    st.error(f"Gagal! ID {id_terpilih} tidak ditemukan di database.")
                st.session_state["konfirmasi_hapus"] = False
                st.session_state["id_akan_dihapus"] = None
                st.cache_data.clear()
                st.rerun()
        with col_tidak:
            if st.button("✗ Batal"):
                st.session_state["konfirmasi_hapus"] = False
                st.session_state["id_akan_dihapus"] = None
                st.rerun()

# ── Halaman: Ringkasan ─────────────────────────────────────────────────────────
def halaman_ringkasan(anggaran):
    st.subheader("Ringkasan Pengeluaran")
    col1, col2 = st.columns([1, 2])
    with col1:
        pilihan = st.selectbox("Filter Periode:", ["Semua Waktu", "Hari Ini", "Pilih Tanggal"])

    tanggal_filter = None
    label = "(Semua Waktu)"
    if pilihan == "Hari Ini":
        tanggal_filter = datetime.date.today()
        label = f"({tanggal_filter.strftime('%d %b %Y')})"
    elif pilihan == "Pilih Tanggal":
        tanggal_filter = st.date_input("Pilih Tanggal:", value=datetime.date.today())
        label = f"({tanggal_filter.strftime('%d %b %Y')})"

    with col2:
        total = anggaran.hitung_total_pengeluaran(tanggal=tanggal_filter)
        st.metric(label=f"Total Pengeluaran {label}", value=format_rp(total))

    st.divider()
    st.subheader(f"Pengeluaran per Kategori {label}")
    dict_kat = anggaran.get_pengeluaran_per_kategori(tanggal=tanggal_filter)

    if not dict_kat:
        st.info("Tidak ada data untuk periode ini.")
    else:
        df_kat = pd.DataFrame(
            [{"Kategori": k, "Total": v} for k, v in dict_kat.items()]
        ).sort_values("Total", ascending=False).reset_index(drop=True)
        df_kat["Total (Rp)"] = df_kat["Total"].apply(format_rp)

        c1, c2 = st.columns(2)
        with c1:
            st.write("Tabel:")
            st.dataframe(df_kat[["Kategori", "Total (Rp)"]], hide_index=True,
                         use_container_width=True)
        with c2:
            st.write("Grafik:")
            st.bar_chart(df_kat.set_index("Kategori")["Total"], use_container_width=True)

# ── Main ───────────────────────────────────────────────────────────────────────
def main():
    st.sidebar.title("Catatan Pengeluaran")
    menu = st.sidebar.radio("Pilih Menu:", ["Tambah", "Riwayat", "Ringkasan"])
    st.sidebar.markdown("---")
    st.sidebar.info("Jobsheet 11 - Aplikasi Keuangan OOP")

    manajer = get_anggaran_manager()
    if menu == "Tambah":
        halaman_input(manajer)
    elif menu == "Riwayat":
        halaman_riwayat(manajer)
    elif menu == "Ringkasan":
        halaman_ringkasan(manajer)

    st.markdown("---")
    st.caption("Pengembangan Aplikasi Berbasis OOP - Politeknik Negeri Semarang")

if __name__ == "__main__":
    main()

Overwriting main_app.py


#Langkah 6: Menjalankan dan Menguji Aplikasi Modular

Setup Database

In [22]:
import database
database.setup_database_initial()
print("Database siap!")

Memeriksa/membuat tabel di database: /content/pengeluaran_harian.db
-> Tabel 'transaksi' siap.
Database siap!


Menjalankan Streamlit + Ngrok

In [26]:
# Hentikan proses lama
!pkill -f streamlit
!pkill -f ngrok

import time
time.sleep(3)
print("Siap. Jalankan ulang sel ngrok.")

Siap. Jalankan ulang sel ngrok.


In [27]:
import subprocess
import threading
import time
import requests
from pyngrok import ngrok
from google.colab import userdata

# Install jika belum ada
!pip install pyngrok streamlit -q

NGROK_AUTHTOKEN = userdata.get('NGROK_AUTHTOKEN')
ngrok.set_auth_token(NGROK_AUTHTOKEN)

# Matikan tunnel lama jika ada
ngrok.kill()

# Jalankan Streamlit di background
def run_streamlit():
    subprocess.run([
        "streamlit", "run", "main_app.py",
        "--server.port", "8501",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
        "--server.address", "0.0.0.0"   # ← tambahan penting
    ])

thread = threading.Thread(target=run_streamlit, daemon=True)
thread.start()

# Tunggu sampai Streamlit benar-benar siap
print("Menunggu Streamlit siap", end="")
for i in range(30):  # coba selama max 30 detik
    time.sleep(1)
    print(".", end="", flush=True)
    try:
        resp = requests.get("http://localhost:8501", timeout=1)
        if resp.status_code == 200:
            print("\nStreamlit sudah siap!")
            break
    except:
        continue
else:
    print("\nPeringatan: Streamlit mungkin belum siap, coba tetap lanjut...")

# Buat tunnel ngrok
public_url = ngrok.connect(8501)
print("=" * 50)
print(f"APLIKASI BISA DIAKSES DI:")
print(f"{public_url}")
print("=" * 50)

Menunggu Streamlit siap..
Streamlit sudah siap!
APLIKASI BISA DIAKSES DI:
NgrokTunnel: "https://passcode-reveal-sacrament.ngrok-free.dev" -> "http://localhost:8501"
